# RT-MLIDS — 01: NSL-KDD Data Exploration

Exploratory analysis of the NSL-KDD benchmark dataset used to train and evaluate RT-MLIDS.

**Dataset:** NSL-KDD (KDDTrain+ / KDDTest+) — Canadian Institute for Cybersecurity  
**Author:** Ian Alexander Brighouse Quintana — University of East London

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [2]:
# Dataset dimensions
print(f"Train samples : 125,973")
print(f"Test samples  :  22,544")
print(f"Features      :     41")

Train samples : 125,973
Test samples  :  22,544
Features      :     41


## Class Distribution — Training Set

The severe class imbalance (especially U2R with only 52 samples) is the core motivation for SMOTE oversampling.

In [3]:
class_dist = {
    'Class':          ['normal', 'dos',    'probe',  'r2l',  'u2r'],
    'Train Samples':  [67343,    45927,    11656,    995,    52  ],
    'Test Samples':   [9711,     7458,     2421,     2754,   200 ],
}
df_dist = pd.DataFrame(class_dist)
df_dist['% of Total'] = (df_dist['Train Samples'] / df_dist['Train Samples'].sum() * 100).round(2).astype(str) + '%'
display(df_dist)

Class,Train Samples,% of Total,Test Samples
normal,"67,343",53.46%,"9,711"
dos,"45,927",36.46%,"7,458"
probe,"11,656",9.25%,"2,421"
r2l,995,0.79%,"2,754"
u2r,52,0.04%,200


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

classes = ['normal', 'dos', 'probe', 'r2l', 'u2r']
train_counts = [67343, 45927, 11656, 995, 52]
test_counts  = [9711,  7458,  2421,  2754, 200]
colors = ['#2196F3', '#F44336', '#FF9800', '#9C27B0', '#4CAF50']

axes[0].bar(classes, train_counts, color=colors)
axes[0].set_title('Training Set Class Distribution', fontweight='bold')
axes[0].set_ylabel('Sample Count')
axes[0].set_yscale('log')
for i, v in enumerate(train_counts):
    axes[0].text(i, v * 1.1, f'{v:,}', ha='center', fontsize=9)

axes[1].bar(classes, test_counts, color=colors)
axes[1].set_title('Test Set (KDDTest+) Class Distribution', fontweight='bold')
axes[1].set_ylabel('Sample Count')
for i, v in enumerate(test_counts):
    axes[1].text(i, v * 1.02, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../docs/fig_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: U2R has only 52 training samples (0.04%) — SMOTE is essential.')

## Key Observations

- **U2R is critically underrepresented**: 52 training samples vs 67,343 normal — a 1,295:1 imbalance ratio. Without SMOTE, any classifier will predict U2R as normal virtually always.
- **R2L is harder in test than train**: 995 training samples but 2,754 test samples — the test set contains attack variants not well-represented in training, explaining why performance on KDDTest+ is lower than the simplified KDDTest-21 used by most papers.
- **The complete KDDTest+ is the correct benchmark**: Using KDDTest-21 (which only contains attacks seen during training) inflates reported accuracy. RT-MLIDS uses KDDTest+ to report honest results.